In [54]:
import pandas as pd

In [ ]:
train_df = pd.read_csv('../../data/processed/train.csv')
val_df = pd.read_csv('../../data/processed/val.csv')
test_df = pd.read_csv('../../data/processed/test.csv')
train_df.shape, val_df.shape, test_df.shape

((172064, 65), (43016, 65), (53770, 65))

In [56]:
import pandas as pd

df = pd.concat([train_df, val_df, test_df], axis=0, ignore_index=True)
df.shape

(268850, 65)

In [57]:
drop_cols = ['serviceCharge', 'picturecount', 'yearConstructed', 'pricetrend', 'baseRent', 'regio2_freq', 'numberOfFloors', 'floor']
df.drop(columns=drop_cols, inplace=True)


In [58]:
df['rent_level'] = pd.qcut(df['totalRent'], q=3, labels=['low','medium','high'])
df['space_level'] = pd.qcut(df['livingSpace'], q=3, labels=['small','medium','large'])
df['room_level'] = pd.qcut(df['noRooms'], q=3, labels=['few','medium','many'])
# One-hot encode
df = pd.get_dummies(df, columns=['rent_level','space_level','room_level'])

df.drop(columns=['totalRent', 'livingSpace', 'noRooms'], inplace=True)


In [59]:
df = df.astype('bool')
df.dtypes

newlyConst            bool
balcony               bool
hasKitchen            bool
cellar                bool
lift                  bool
                      ... 
space_level_medium    bool
space_level_large     bool
room_level_few        bool
room_level_medium     bool
room_level_many       bool
Length: 63, dtype: object

In [62]:
from mlxtend.frequent_patterns import apriori

frequent_itemsets = apriori(df, 
                           min_support=0.05,   # có thể chỉnh
                           use_colnames=True)

frequent_itemsets = frequent_itemsets.sort_values(by='support', ascending=False)
frequent_itemsets

,support,itemsets
3,0.640636,(cellar)
1,0.616455,(balcony)
33,0.447298,(room_level_few)
39,0.435555,"(balcony, cellar)"
34,0.408555,(room_level_medium)
...,...,...
368,0.050351,"(balcony, rent_level_medium, cellar, lift)"
234,0.050288,"(balcony, condition_unknown, rent_level_medium)"
349,0.050218,"(typeOfFlat_unknown, room_level_few, space_lev..."
68,0.050199,"(regio1_nordrhein_westfalen, hasKitchen)"


In [74]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(frequent_itemsets, 
                          metric="confidence", 
                          min_threshold=0.6)

rules = rules[
    (rules['lift'] > 2) &
    (rules['confidence'] > 0.7) &
    (rules['support'] > 0.05)
]

rules = rules.sort_values(by='lift', ascending=False)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
171,"(regio1_sachsen, space_level_small)","(room_level_few, rent_level_low)",0.094298,0.225933,0.072405,0.767829,3.398486,1.0,0.051100,3.334039,0.779231,0.292159,0.700064,0.544150
346,"(balcony, newlyConst)",(lift),0.068034,0.235529,0.050147,0.737084,3.129481,1.0,0.034123,2.907660,0.730132,0.197883,0.656081,0.474998
279,(newlyConst),(lift),0.078747,0.235529,0.055563,0.705588,2.995756,1.0,0.037015,2.596601,0.723139,0.214765,0.614881,0.470747
328,"(balcony, rent_level_high, room_level_many)",(space_level_large),0.058267,0.333323,0.051627,0.886052,2.658234,1.0,0.032206,5.850694,0.662407,0.151862,0.829080,0.520469
201,"(rent_level_high, room_level_many)",(space_level_large),0.074302,0.333323,0.065817,0.885813,2.657518,1.0,0.041051,5.838462,0.673771,0.192557,0.828722,0.541635
311,"(room_level_few, hasKitchen, rent_level_low)",(space_level_small),0.061350,0.333465,0.052933,0.862799,2.587376,1.0,0.032475,4.858079,0.653607,0.154828,0.794157,0.510767
303,"(room_level_few, condition_unknown, rent_level...",(space_level_small),0.063028,0.333465,0.053327,0.846090,2.537271,1.0,0.032310,4.330690,0.646631,0.155398,0.769090,0.503004
299,"(regio1_sachsen, cellar, space_level_small)",(rent_level_low),0.062752,0.336939,0.053450,0.851757,2.527929,1.0,0.032306,4.472813,0.644888,0.154372,0.776427,0.505196
167,"(room_level_few, regio1_sachsen, space_level_s...",(rent_level_low),0.085557,0.336939,0.072405,0.846274,2.511656,1.0,0.043577,4.313273,0.658167,0.206817,0.768158,0.530582
139,"(regio1_sachsen, space_level_small)",(rent_level_low),0.094298,0.336939,0.079401,0.842024,2.499042,1.0,0.047629,4.197235,0.662300,0.225677,0.761748,0.538839


In [69]:
rules_support = rules.sort_values(by='support', ascending=False)
rules_support

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(balcony),(cellar),0.616455,0.640636,0.435555,0.706548,1.102885,1.0,0.040632,1.224608,0.243224,0.530172,0.183412,0.693214
1,(cellar),(balcony),0.640636,0.616455,0.435555,0.679879,1.102885,1.0,0.040632,1.198125,0.259589,0.530172,0.165363,0.693214
2,(room_level_few),(space_level_small),0.447298,0.333465,0.290902,0.650354,1.950294,1.0,0.141744,1.906316,0.881590,0.593847,0.475428,0.761358
3,(space_level_small),(room_level_few),0.333465,0.447298,0.290902,0.872362,1.950294,1.0,0.141744,4.330234,0.731029,0.593847,0.769066,0.761358
4,(room_level_medium),(balcony),0.408555,0.616455,0.277757,0.679853,1.102842,1.0,0.025901,1.198025,0.157667,0.371704,0.165293,0.565212
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
344,"(typeOfFlat_unknown, room_level_few)",(space_level_small),0.069942,0.333465,0.050218,0.717986,2.153108,1.0,0.026894,2.363479,0.575830,0.142183,0.576895,0.434289
345,"(typeOfFlat_unknown, space_level_small)",(room_level_few),0.057050,0.447298,0.050218,0.880232,1.967889,1.0,0.024699,4.614778,0.521599,0.110580,0.783305,0.496250
346,"(balcony, newlyConst)",(lift),0.068034,0.235529,0.050147,0.737084,3.129481,1.0,0.034123,2.907660,0.730132,0.197883,0.656081,0.474998
347,"(newlyConst, lift)",(balcony),0.055563,0.616455,0.050147,0.902530,1.464065,1.0,0.015895,3.935021,0.335618,0.080639,0.745872,0.491939
